# 02 Model

Google Colab notebook version.

In [ ]:

# ============================================================
# MODEL: Softmax Temperature Attention + CNN-BiLSTM-STAM
# ============================================================

import tensorflow as tf
from tensorflow.keras.layers import (
    Layer, Dense, Input, Bidirectional, LSTM, Conv1D,
    SpatialDropout1D, LayerNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.losses import Huber

class SoftmaxTemperatureAttention(Layer):
    def __init__(self, attn_dim, tau_init=1.0, decay=0.997, **kwargs):
        super().__init__(**kwargs)
        self.attn_dim = attn_dim
        self.tau_init = tau_init
        self.decay = decay

    def build(self, input_shape):
        self.tau = self.add_weight(
            shape=(),
            initializer=tf.constant_initializer(self.tau_init),
            trainable=True,
            name="softmax_tau",
        )
        self.global_dense = Dense(self.attn_dim, activation="tanh")
        self.attn_dense = Dense(input_shape[-1])
        super().build(input_shape)

    def call(self, x, training=None):
        ctx = tf.reduce_mean(x, axis=1, keepdims=True)
        ctx_proj = self.global_dense(ctx)
        ctx_tiled = tf.tile(ctx_proj, [1, tf.shape(x)[1], 1])

        concat = tf.concat([x, ctx_tiled], axis=-1)
        logits = self.attn_dense(concat)

        temp = tf.math.maximum(self.tau, 1e-4)
        attn_w = tf.nn.softmax(logits / temp, axis=1)
        attended = tf.reduce_sum(x * attn_w, axis=1)

        if training is True:
            self.tau.assign(self.tau * self.decay)

        return attended

def build_cnn_bilstm_stam(
    window,
    n_features,
    conv_filters=64,
    kernel_size=3,
    lstm_units_1=64,
    lstm_units_2=32,
    dropout=0.30,
    attn_dim=64,
    tau_init=1.0,
    tau_decay=0.997,
    learning_rate=1e-4,
    huber_delta=1.0,
):
    inp = Input(shape=(window, n_features))

    x = Conv1D(conv_filters, kernel_size, activation="relu", padding="same")(inp)
    x = Conv1D(conv_filters, kernel_size, activation="relu", padding="same")(x)
    x = LayerNormalization()(x)

    x = Bidirectional(LSTM(lstm_units_1, return_sequences=True))(x)
    x = SpatialDropout1D(dropout)(x)

    x = Bidirectional(LSTM(lstm_units_2, return_sequences=True))(x)
    x = LayerNormalization()(x)

    attn = SoftmaxTemperatureAttention(
        attn_dim=attn_dim,
        tau_init=tau_init,
        decay=tau_decay,
    )(x)

    out = Dense(1)(attn)
    model = Model(inp, out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=Huber(delta=huber_delta),
        metrics=[tf.keras.metrics.RootMeanSquaredError(name="RMSE")],
    )

    return model


In [ ]:
# Example model build
# Run preprocessing first if you want to use len(FEATURES)
WINDOW = 48
N_FEATURES = 17  # change if needed

model = build_cnn_bilstm_stam(window=WINDOW, n_features=N_FEATURES)
model.summary()